# Train and evaluate on Kaggle

Runs the whole project on the DFDC sample that ships with the
[Deepfake Detection Challenge](https://www.kaggle.com/competitions/deepfake-detection-challenge/data)
(400 labelled clips).

Before running:

1. Add the competition data to the notebook (**Add Input → Competitions → Deepfake Detection Challenge**). You need to accept the competition rules once.
2. Settings: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
3. **Run All.** Face extraction takes most of the time; training is a few minutes on a GPU.

Outputs land in `/kaggle/working/runs/efficientnet_b0`: the model, `metrics.json`, `report.md` and the two plots.

In [ ]:
BRANCH = "v2"
!rm -rf /kaggle/working/repo
!git clone -q --depth 1 -b $BRANCH https://github.com/Marwankhater/DEEP-FAKE-DETECTION.git /kaggle/working/repo
# Kaggle already has TensorFlow. The code is imported straight from repo/src, so only MTCNN is needed.
!pip install -q "mtcnn>=1.0" lz4 pyyaml


In [ ]:
import os, sys
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "src")

import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Split by original video and extract faces

In [ ]:
from deepfake_detector.config import load_config
from deepfake_detector.extract import extract

config = load_config("configs/kaggle.yaml")
index = extract(config)
index.groupby(["split", "label"]).size().unstack()

## 2. Train

In [ ]:
from deepfake_detector.train import train

model_path = train(config)

In [ ]:
import pandas as pd
history = pd.read_csv(f"{config.run_dir}/history.csv")
history[["epoch", "auc", "val_auc", "loss", "val_loss"]].round(3)

## 3. Evaluate on the held-out test videos

In [ ]:
from IPython.display import Image, Markdown, display
from deepfake_detector.evaluate import evaluate

results = evaluate(config)
display(Markdown(open(f"{config.run_dir}/report.md").read()))
display(Image(f"{config.run_dir}/roc.png"), Image(f"{config.run_dir}/confusion.png"))

## 4. Save the results

In [ ]:
!cd /kaggle/working && zip -q -r results.zip runs/efficientnet_b0 faces/videos.csv
print("Download /kaggle/working/results.zip from the Output panel.")